# 📊 Bank Marketing Optimization Project
## Business-Focused Customer Targeting with Logistic Regression

### Project Goal
The purpose of this project is to help the bank reduce wasted calls and improve campaign performance by identifying customers with the highest likelihood of subscribing to a term deposit.

### Business Problem
The bank currently spends time and money contacting many customers, but only a small proportion subscribe. A data-driven targeting approach can help focus effort on the right customers.

### Business Questions
1. Which customer groups are more likely to subscribe?
2. Which campaign patterns reduce success?
3. How can the bank prioritize leads to improve conversion and reduce cost?


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve

import warnings
warnings.filterwarnings("ignore")

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)


## 2. Load the Dataset

In [ ]:
df = pd.read_csv("/mnt/data/bank-additional-full.csv", sep=";")
df.head()


## 3. Initial Data Review

We first inspect the data structure, variable types, and target balance to understand the quality of the dataset and the business challenge.


In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.describe(include="all").T.head(10))


## 4. Target Variable Distribution

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(data=df, x="y")
plt.title("Subscription Outcome Distribution")
plt.xlabel("Subscribed to Term Deposit")
plt.ylabel("Number of Customers")
plt.show()

target_share = df["y"].value_counts(normalize=True).mul(100).round(2)
target_share


### Business Interpretation
The target distribution shows that **most customers did not subscribe**. This means the bank is operating in a **low-conversion environment**, where calling everyone is inefficient.

**Business conclusion:**  
A selective campaign is necessary. Even a simple model that helps the bank concentrate on higher-probability customers can create substantial operational savings.


## 5. Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x="job", order=df["job"].value_counts().index, ax=axes[0,0])
axes[0,0].set_title("Customers by Job")
axes[0,0].tick_params(axis="x", rotation=60)

sns.countplot(data=df, x="marital", order=df["marital"].value_counts().index, ax=axes[0,1])
axes[0,1].set_title("Customers by Marital Status")

sns.countplot(data=df, x="education", order=df["education"].value_counts().index, ax=axes[1,0])
axes[1,0].set_title("Customers by Education")
axes[1,0].tick_params(axis="x", rotation=60)

sns.boxplot(data=df, x="y", y="age", ax=axes[1,1])
axes[1,1].set_title("Age Distribution by Subscription Outcome")

plt.tight_layout()
plt.show()


### Business Interpretation
This exploration helps the bank understand the composition of its customer base. Differences across job, age, education, and marital status can reveal where stronger response patterns exist.

**Business conclusion:**  
The bank should not treat all customers equally. Marketing performance can improve when customer outreach is adapted to segment characteristics rather than using one broad campaign for everyone.


## 6. Remove a Leakage Variable

In [ ]:
df_model = df.copy()
df_model = df_model.drop("duration", axis=1)
print("Remaining columns:", df_model.shape[1])


### Why `duration` was removed
The variable `duration` is the length of the call. It is highly predictive, but it is only known **after** the call has happened. Using it would make the model look stronger than it would be in real operations.

**Business conclusion:**  
Removing `duration` makes the solution more realistic and more useful for the marketing team, because it ensures the model can be used **before** the bank decides whom to call.


## 7. Encode Categorical Variables

In [ ]:
encoders = {}
for col in df_model.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    encoders[col] = le

df_model.head()


## 8. Correlation Heatmap

In [ ]:
plt.figure(figsize=(14,10))
sns.heatmap(df_model.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()


### Business Interpretation
The correlation heatmap gives an early view of which variables move together. It also helps detect whether some campaign or macroeconomic factors may be associated with subscription behavior.

**Business conclusion:**  
The decision to subscribe is not driven by a single feature. It is influenced by a combination of customer profile, campaign history, and broader economic context. This supports using a predictive model instead of simple rules.


## 9. Prepare Data for Modeling

In [ ]:
X = df_model.drop("y", axis=1)
y = df_model["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)


## 10. Train Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=2000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:,1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("Accuracy:", round(accuracy, 4))
print("ROC-AUC :", round(auc, 4))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


### Business Interpretation
Accuracy alone does not tell the full story in an imbalanced marketing dataset, so ROC-AUC is included to assess ranking quality. Logistic Regression is valuable here because it is transparent and practical for business use.

**Business conclusion:**  
This model should be used primarily as a **prioritization tool**, not just a yes/no classifier. Its biggest business value comes from helping the bank rank customers by likelihood and concentrate effort where it matters most.


## 11. Confusion Matrix

In [ ]:
plt.figure(figsize=(6,4))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


### Business Interpretation
The confusion matrix shows how many customers were correctly or incorrectly classified.

- **True negatives:** customers correctly identified as unlikely to subscribe  
- **True positives:** customers correctly identified as likely to subscribe  
- **False positives:** customers contacted unnecessarily  
- **False negatives:** customers missed by the model  

**Business conclusion:**  
The cost of false positives is wasted call effort, while the cost of false negatives is missed revenue. The bank can later adjust the probability threshold depending on whether it wants to prioritize **cost reduction** or **maximum lead capture**.


## 12. ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc:.3f})")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


### Business Interpretation
The ROC curve shows the trade-off between capturing more likely subscribers and increasing false alarms.

**Business conclusion:**  
This is useful for management decisions. If the bank wants a lean campaign, it can use a stricter threshold. If it wants to pursue growth aggressively, it can lower the threshold and contact more customers.


## 13. Coefficient Analysis

In [ ]:
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=False)

display(coef_df.head(10))
display(coef_df.tail(10))


### Business Interpretation of Coefficients
Positive coefficients increase the likelihood of subscription; negative coefficients decrease it. Because the data was scaled, the coefficients can be compared more fairly.

**Business conclusion:**  
This section is especially important for management because it shows **why** the model makes decisions. It helps transform analytics into action:

- Features with strong positive influence suggest where the bank should focus.
- Features with strong negative influence highlight where contact strategy may be less efficient.


## 14. Customer Probability Scoring

In [ ]:
X_scaled_full = scaler.transform(X)
df_scored = df.copy()
df_scored["subscription_probability"] = model.predict_proba(X_scaled_full)[:,1]

df_scored["customer_segment"] = pd.cut(
    df_scored["subscription_probability"],
    bins=[0, 0.30, 0.60, 1.00],
    labels=["Low Value", "Medium Value", "High Value"],
    include_lowest=True
)

df_scored[["subscription_probability", "customer_segment"]].head()


## 15. Segment Distribution

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(data=df_scored, x="customer_segment", order=["Low Value", "Medium Value", "High Value"])
plt.title("Customer Segment Distribution")
plt.xlabel("Segment")
plt.ylabel("Number of Customers")
plt.show()

df_scored["customer_segment"].value_counts()


### Business Interpretation
Segmenting customers into Low, Medium, and High Value makes the model easy to operationalize.

**Business conclusion:**  
The bank can use the segments immediately:

- **High Value:** call first, assign best agents, use strongest offers  
- **Medium Value:** contact if budget remains or use lower-cost channels  
- **Low Value:** deprioritize to save resources  

This converts the model into a direct business action plan.


## 16. Top Priority Customer List

In [ ]:
priority_customers = df_scored.sort_values("subscription_probability", ascending=False).copy()
priority_customers.to_csv("/mnt/data/priority_customers.csv", index=False)

priority_customers.head(20)


### Business Interpretation
This table is the most actionable output of the project. It gives the marketing team a ranked list of customers to target first.

**Business conclusion:**  
Instead of calling the entire customer base, the bank can focus on the highest-ranked customers. This supports:

- lower campaign cost  
- higher conversion efficiency  
- better use of sales staff time  
- more disciplined and measurable marketing execution


## 17. Segment-Level Business Profile

In [ ]:
segment_summary = df_scored.groupby("customer_segment").agg(
    avg_age=("age", "mean"),
    avg_campaign_contacts=("campaign", "mean"),
    avg_previous_contacts=("previous", "mean"),
    avg_probability=("subscription_probability", "mean")
).round(2)

segment_summary


### Business Interpretation
This summary compares segment behavior.

**Business conclusion:**  
If the High Value segment shows fewer campaign contacts but higher predicted probability, it suggests that over-contacting is inefficient and that better lead selection matters more than increasing volume.


## 18. Final Business Conclusions

### Main Findings
1. The subscription rate is low, so mass calling is inefficient.  
2. A realistic model can be built without call duration and still provide useful targeting support.  
3. The model is most valuable for **ranking and segmentation**, not just binary prediction.  
4. Customer prioritization can help the bank reduce waste and improve campaign productivity.

### Recommended Business Actions
- Use the scored customer list before each campaign.
- Prioritize High Value customers first.
- Limit repeated calls when campaign counts become too high.
- Use Medium Value customers as a second wave.
- Avoid spending full call-center effort on Low Value customers.

### Expected Business Impact
By moving from broad calling to targeted calling, the bank can:
- reduce marketing cost
- improve conversion rate
- increase return on campaign effort
- make campaign planning more measurable and repeatable

### Final Recommendation to Management
This project should be adopted as a **campaign targeting support tool**. Logistic Regression is simple, explainable, and operationally practical. It gives the bank a clear way to decide **who to call first**, which is the core business problem this project was designed to solve.
